# Hotel Bookings Analytics Platform - Complete Pipeline

This notebook contains the complete, end-to-end data pipeline for the Hotel Bookings Analytics project. It combines data cleaning, exploratory data analysis, SQL analytics, machine learning, and explainability into a single executable notebook.

## Phase: data_cleaning.py
Executing code from `data_cleaning.py`

In [ ]:
"""
data_cleaning.py
================
Phase 2 — Data Cleaning & Preprocessing Pipeline
Hotel Bookings Dataset (hotel_bookings.csv)

Preprocessing decisions are documented inline and summarised in
reports/PREPROCESSING_DECISIONS.md.

Usage:
    python src/data_cleaning.py

Output:
    data/processed/hotel_bookings_cleaned.csv   — cleaned dataset
    reports/cleaning_report.txt                 — row-level audit log
"""

import sys
import textwrap
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Suppress pandas FutureWarning for replace() downcasting (resolved in pandas 3.x)
pd.set_option("future.no_silent_downcasting", True)

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------

ROOT        = Path.cwd()
INPUT_FILE  = ROOT / "data" / "raw" / "hotel_bookings.csv"
OUTPUT_FILE = ROOT / "data" / "processed" / "hotel_bookings_cleaned.csv"
REPORT_FILE = ROOT / "reports" / "cleaning_report.txt"

# ADR cap: values above this are treated as extreme outliers.
# The single row with ADR=5400 (canceled, Non Refund) is 53× the mean;
# 5400 is retained as flagged rather than dropped, but values above
# this hard cap are removed (none exist beyond 5400, so this is a safety net).
ADR_UPPER_CAP = 5400.0

# ---------------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------------

def section(title: str, width: int = 70) -> str:
    bar = "=" * width
    return f"\n{bar}\n  {title}\n{bar}"


def log(report_lines: list, msg: str) -> None:
    safe = msg.encode(sys.stdout.encoding or "utf-8", errors="replace").decode(
        sys.stdout.encoding or "utf-8"
    )
    print(safe)
    report_lines.append(msg)


# ---------------------------------------------------------------------------
# STEP 1 — LOAD RAW DATA
# ---------------------------------------------------------------------------

def load_raw(path: str, report: list) -> pd.DataFrame:
    log(report, section("STEP 1 — LOAD RAW DATA"))
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    log(report, f"  Rows loaded : {len(df):,}")
    log(report, f"  Columns     : {df.shape[1]}")
    log(report, f"  File        : {path}")
    return df


# ---------------------------------------------------------------------------
# STEP 2 — SCHEMA VALIDATION (expected columns & raw types)
# ---------------------------------------------------------------------------

EXPECTED_COLUMNS = [
    "hotel", "is_canceled", "lead_time", "arrival_date_year",
    "arrival_date_month", "arrival_date_week_number",
    "arrival_date_day_of_month", "stays_in_weekend_nights",
    "stays_in_week_nights", "adults", "children", "babies", "meal",
    "country", "market_segment", "distribution_channel",
    "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled", "reserved_room_type",
    "assigned_room_type", "booking_changes", "deposit_type", "agent",
    "company", "days_in_waiting_list", "customer_type", "adr",
    "required_car_parking_spaces", "total_of_special_requests",
    "reservation_status", "reservation_status_date",
]


def validate_schema(df: pd.DataFrame, report: list) -> pd.DataFrame:
    log(report, section("STEP 2 — SCHEMA VALIDATION"))

    missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    extra_cols   = [c for c in df.columns if c not in EXPECTED_COLUMNS]

    if missing_cols:
        log(report, f"  MISSING columns : {missing_cols}")
        sys.exit("Schema error — missing expected columns. Aborting.")
    if extra_cols:
        log(report, f"  EXTRA columns (will be kept) : {extra_cols}")
    else:
        log(report, "  All 32 expected columns present. ✓")

    # Enforce canonical column order
    df = df[EXPECTED_COLUMNS + extra_cols]
    return df


# ---------------------------------------------------------------------------
# STEP 3 — REPLACE "NULL" STRINGS WITH REAL NaN
# ---------------------------------------------------------------------------

def replace_null_strings(df: pd.DataFrame, report: list) -> pd.DataFrame:
    """
    Decision: The CSV stores missing values as the literal string "NULL"
    for `agent` and `company`. Replace with NaN so downstream logic
    treats them as missing.
    """
    log(report, section("STEP 3 — REPLACE 'NULL' STRINGS WITH NaN"))
    # Replace "NULL" strings globally — the CSV uses this as a missing-value
    # marker for agent, company, and country.
    before_null_str = df.isna().sum().sum()
    df = df.replace("NULL", np.nan).infer_objects(copy=False)
    after_null_str = df.isna().sum().sum()
    log(report, f"  'NULL' string → NaN conversions (all columns): {after_null_str - before_null_str:,}")

    # Also replace empty strings with NaN across all columns
    before = df.isna().sum().sum()
    df = df.replace("", np.nan).infer_objects(copy=False)
    after  = df.isna().sum().sum()
    log(report, f"  Empty string → NaN conversions: {after - before:,}")
    return df


# ---------------------------------------------------------------------------
# STEP 4 — CAST DATA TYPES
# ---------------------------------------------------------------------------

INTEGER_COLS = [
    "is_canceled", "lead_time", "arrival_date_year",
    "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "babies", "is_repeated_guest",
    "previous_cancellations", "previous_bookings_not_canceled",
    "booking_changes", "days_in_waiting_list",
    "required_car_parking_spaces", "total_of_special_requests",
]

FLOAT_COLS = ["adr"]

CATEGORICAL_COLS = [
    "hotel", "arrival_date_month", "meal", "country", "market_segment",
    "distribution_channel", "reserved_room_type", "assigned_room_type",
    "deposit_type", "customer_type", "reservation_status",
]


def cast_dtypes(df: pd.DataFrame, report: list) -> pd.DataFrame:
    """
    Decision: children is cast to float (not int) because it contains
    NA strings that are converted to NaN, and pandas int cannot hold NaN.
    agent and company are kept as string (they are IDs / labels).
    """
    log(report, section("STEP 4 — CAST DATA TYPES"))

    # children: coerce 'NA' and '10' alongside numerics
    df["children"] = pd.to_numeric(df["children"], errors="coerce")
    log(report, "  children  → float64 (NA strings → NaN; '10' kept as numeric)")

    for col in INTEGER_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
        log(report, f"  {col}  → Int64")

    for col in FLOAT_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        log(report, f"  {col}  → float64")

    for col in CATEGORICAL_COLS:
        df[col] = df[col].astype("category")
        log(report, f"  {col}  → category")

    # Parse reservation_status_date as date
    df["reservation_status_date"] = pd.to_datetime(
        df["reservation_status_date"], errors="coerce"
    )
    log(report, "  reservation_status_date → datetime64")

    return df


# ---------------------------------------------------------------------------
# STEP 5 — HANDLE MISSING VALUES
# ---------------------------------------------------------------------------

def handle_missing(df: pd.DataFrame, report: list) -> pd.DataFrame:
    """
    Decisions:
    - country (488 nulls, 0.41%): fill with 'Unknown' — small fraction,
      dropping rows would lose valid booking data.
    - agent (16,340 nulls, 13.7%): fill with 0 (sentinel for 'no agent') —
      consistent with the dataset convention used in published research.
    - company (112,593 nulls, 94.3%): fill with 0 (sentinel for 'no company')
      — nearly all rows are null; 0 = "direct / no company affiliation".
    - children (4 nulls): fill with 0 — only 4 rows; 0 is the modal value.
    """
    log(report, section("STEP 5 — HANDLE MISSING VALUES"))

    fills = {
        "country":  "Unknown",
        "agent":    "0",
        "company":  "0",
        "children": 0,
    }
    for col, fill_val in fills.items():
        n = df[col].isna().sum()
        # Categorical columns need the new value added to their category list first
        if hasattr(df[col], "cat"):
            if fill_val not in df[col].cat.categories:
                df[col] = df[col].cat.add_categories([fill_val])
        df[col] = df[col].fillna(fill_val)
        log(report, f"  {col}: {n:,} nulls filled with {repr(fill_val)}")

    after_nulls = df.isna().sum()
    remaining   = after_nulls[after_nulls > 0]
    if len(remaining):
        log(report, f"  Remaining nulls:\n{remaining.to_string()}")
    else:
        log(report, "  No remaining nulls after filling. ✓")

    return df


# ---------------------------------------------------------------------------
# STEP 6 — INVALID / ANOMALOUS ROW REMOVAL
# ---------------------------------------------------------------------------

def remove_invalid_rows(df: pd.DataFrame, report: list) -> pd.DataFrame:
    """
    Decisions:
    - Zero-night stays (both stays_in_weekend_nights AND stays_in_week_nights
      = 0): 715 rows. These represent bookings with no stay duration and are
      logically invalid for demand / revenue analysis. Removed.
    - Zero-adult bookings (adults = 0 AND children = 0 AND babies = 0):
      no legitimate reservation can have zero guests. Removed.
    - Rows where ADR is negative: 1 row (ADR = -6.38, Resort Hotel,
      Check-Out). Negative revenue is not meaningful. Removed.
    - ADR > ADR_UPPER_CAP: safety net removal of extreme ADR outliers
      (the ADR=5400 row is a Canceled booking and is retained; values beyond
      the hard cap would be removed if they existed).
    """
    log(report, section("STEP 6 — INVALID / ANOMALOUS ROW REMOVAL"))
    initial = len(df)

    # Zero-night stays
    mask_zero_nights = (df["stays_in_weekend_nights"] == 0) & (df["stays_in_week_nights"] == 0)
    n_zero_nights = mask_zero_nights.sum()
    df = df[~mask_zero_nights].copy()
    log(report, f"  Removed zero-night stays        : {n_zero_nights:,} rows")

    # Zero-guest bookings
    mask_zero_guests = (df["adults"] == 0) & (df["children"].fillna(0) == 0) & (df["babies"] == 0)
    n_zero_guests = mask_zero_guests.sum()
    df = df[~mask_zero_guests].copy()
    log(report, f"  Removed zero-guest bookings     : {n_zero_guests:,} rows")

    # Negative ADR
    mask_neg_adr = df["adr"] < 0
    n_neg_adr = mask_neg_adr.sum()
    df = df[~mask_neg_adr].copy()
    log(report, f"  Removed negative ADR rows       : {n_neg_adr:,} rows")

    # ADR above hard cap (safety net — no rows expected beyond 5400)
    mask_cap_adr = df["adr"] > ADR_UPPER_CAP
    n_cap_adr = mask_cap_adr.sum()
    if n_cap_adr:
        df = df[~mask_cap_adr].copy()
    log(report, f"  Removed ADR > {ADR_UPPER_CAP} rows        : {n_cap_adr:,} rows")

    removed = initial - len(df)
    log(report, f"  Total rows removed              : {removed:,}")
    log(report, f"  Rows remaining                  : {len(df):,}")
    return df


# ---------------------------------------------------------------------------
# STEP 7 — OUTLIER FLAGGING (non-destructive)
# ---------------------------------------------------------------------------

def flag_outliers(df: pd.DataFrame, report: list) -> pd.DataFrame:
    """
    Decision: Rather than removing high-ADR or extreme lead-time rows
    (which may be genuine premium bookings or long-range planners), we
    add binary flag columns so downstream analysis can filter them.
    No rows are dropped here.
    """
    log(report, section("STEP 7 — OUTLIER FLAGGING (non-destructive)"))

    # ADR outlier: > mean + 3*std
    adr_mean = df["adr"].mean()
    adr_std  = df["adr"].std()
    adr_upper = adr_mean + 3 * adr_std
    df["flag_adr_outlier"] = (df["adr"] > adr_upper).astype("Int64")
    n_adr = df["flag_adr_outlier"].sum()
    log(report, f"  flag_adr_outlier  (ADR > {adr_upper:.2f}): {n_adr:,} rows")

    # Lead time outlier: > 365 days
    df["flag_long_lead_time"] = (df["lead_time"] > 365).astype("Int64")
    n_lt = df["flag_long_lead_time"].sum()
    log(report, f"  flag_long_lead_time (lead_time > 365)   : {n_lt:,} rows")

    # Zero ADR on non-complimentary bookings (possible data quality)
    mask_zero_adr = (df["adr"] == 0) & (~df["market_segment"].isin(["Complementary"]))
    df["flag_zero_adr"] = mask_zero_adr.astype("Int64")
    n_zero = df["flag_zero_adr"].sum()
    log(report, f"  flag_zero_adr (ADR=0, non-compl.)       : {n_zero:,} rows")

    return df


# ---------------------------------------------------------------------------
# STEP 8 — FEATURE ENGINEERING
# ---------------------------------------------------------------------------

MONTH_ORDER = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12,
}


def engineer_features(df: pd.DataFrame, report: list) -> pd.DataFrame:
    """
    New columns added (all derivable from existing fields — no fabrication):

    total_nights          : stays_in_weekend_nights + stays_in_week_nights
    arrival_date          : combined arrival year/month/day as datetime
    arrival_month_num     : integer month (1–12) for sorting/modelling
    room_type_match       : 1 if reserved_room_type == assigned_room_type
    is_high_season        : 1 if arrival month is June, July, or August
    revenue_estimate      : adr × total_nights (proxy for booking value)
    total_guests          : adults + children + babies
    """
    log(report, section("STEP 8 — FEATURE ENGINEERING"))

    # total_nights
    df["total_nights"] = (
        df["stays_in_weekend_nights"].astype(float) +
        df["stays_in_week_nights"].astype(float)
    ).astype("Int64")
    log(report, "  total_nights = stays_in_weekend_nights + stays_in_week_nights")

    # arrival_date
    df["arrival_month_num"] = df["arrival_date_month"].map(MONTH_ORDER).astype("Int64")
    df["arrival_date"] = pd.to_datetime(
        df["arrival_date_year"].astype(str) + "-" +
        df["arrival_month_num"].astype(str).str.zfill(2) + "-" +
        df["arrival_date_day_of_month"].astype(str).str.zfill(2),
        errors="coerce",
    )
    invalid_dates = df["arrival_date"].isna().sum()
    log(report, f"  arrival_date constructed ({invalid_dates} parse failures)")

    # room_type_match
    df["room_type_match"] = (
        df["reserved_room_type"].astype(str) == df["assigned_room_type"].astype(str)
    ).astype("Int64")
    match_rate = df["room_type_match"].mean() * 100
    log(report, f"  room_type_match: {match_rate:.1f}% of bookings got their requested room type")

    # is_high_season (June, July, August)
    df["is_high_season"] = df["arrival_month_num"].isin([6, 7, 8]).astype("Int64")
    n_hs = df["is_high_season"].sum()
    log(report, f"  is_high_season (Jun–Aug): {n_hs:,} rows ({n_hs/len(df)*100:.1f}%)")

    # revenue_estimate
    df["revenue_estimate"] = (df["adr"] * df["total_nights"].astype(float)).round(2)
    log(report, f"  revenue_estimate = adr × total_nights  (mean: {df['revenue_estimate'].mean():.2f})")

    # total_guests
    df["total_guests"] = (
        df["adults"].astype(float) +
        df["children"].astype(float).fillna(0) +
        df["babies"].astype(float)
    ).astype("Int64")
    log(report, f"  total_guests = adults + children + babies  (mean: {df['total_guests'].mean():.2f})")

    return df


# ---------------------------------------------------------------------------
# STEP 9 — FINAL VALIDATION
# ---------------------------------------------------------------------------

def final_validation(df: pd.DataFrame, report: list) -> None:
    log(report, section("STEP 9 — FINAL VALIDATION"))

    log(report, f"  Final row count        : {len(df):,}")
    log(report, f"  Final column count     : {df.shape[1]}")

    nulls = df.isna().sum()
    remaining_nulls = nulls[nulls > 0]
    if len(remaining_nulls):
        log(report, f"  Remaining nulls:\n{remaining_nulls.to_string()}")
    else:
        log(report, "  No unexpected nulls remaining. ✓")

    log(report, f"  is_canceled values     : {dict(df['is_canceled'].value_counts().to_dict())}")
    log(report, f"  ADR range              : {df['adr'].min():.2f} – {df['adr'].max():.2f}")
    log(report, f"  Lead time range        : {df['lead_time'].min()} – {df['lead_time'].max()}")
    log(report, f"  total_nights range     : {df['total_nights'].min()} – {df['total_nights'].max()}")
    log(report, f"  Hotel types            : {dict(df['hotel'].value_counts().to_dict())}")
    log(report, f"  Date range             : {df['arrival_date'].min().date()} – {df['arrival_date'].max().date()}")

    # Confirm no negative values in key numeric columns
    for col in ["adr", "lead_time", "stays_in_weekend_nights", "stays_in_week_nights"]:
        n_neg = (df[col].astype(float) < 0).sum()
        if n_neg:
            log(report, f"  WARNING: {n_neg} negative values in {col}")
        else:
            log(report, f"  {col}: no negative values. ✓")


# ---------------------------------------------------------------------------
# STEP 10 — SAVE OUTPUT
# ---------------------------------------------------------------------------

def save_output(df: pd.DataFrame, path: str, report: list) -> None:
    log(report, section("STEP 10 — SAVE OUTPUT"))
    df.to_csv(path, index=False)
    size_mb = Path(path).stat().st_size / (1024 ** 2)
    log(report, f"  Saved: {path}  ({size_mb:.2f} MB, {len(df):,} rows, {df.shape[1]} columns)")


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

def main():
    report: list = []

    log(report, "Hotel Bookings — Phase 2 Data Cleaning Pipeline")
    log(report, f"pandas {pd.__version__}  |  numpy {np.__version__}")

    df = load_raw(INPUT_FILE, report)
    df = validate_schema(df, report)
    df = replace_null_strings(df, report)
    df = cast_dtypes(df, report)
    df = handle_missing(df, report)
    df = remove_invalid_rows(df, report)
    df = flag_outliers(df, report)
    df = engineer_features(df, report)
    final_validation(df, report)
    save_output(df, str(OUTPUT_FILE), report)

    # Write audit report
    report_text = "\n".join(report)
    REPORT_FILE.write_text(report_text, encoding="utf-8")
    print(f"\nAudit report saved: {REPORT_FILE}")
    print("Pipeline complete.")


if __name__ == "__main__":
    main()


## Phase: eda_analysis.py
Executing code from `eda_analysis.py`

In [ ]:
"""
eda_analysis.py
===============
Phase 3 — Exploratory Data Analysis & KPI Analytics
Hotel Bookings Dataset (hotel_bookings_cleaned.csv)

Sections
--------
  A. Load & Sanity Check
  B. Business KPIs
  C. Cancellation Analysis
  D. Revenue & ADR Analysis
  E. Booking Channel & Market Segment Analysis
  F. Lead Time & Booking Behaviour
  G. Seasonality & Time Trends
  H. Guest Profile Analysis
  I. KPI Summary Report

Outputs
-------
  plots/          — all chart images (PNG, 150 dpi)
  kpi_report.txt  — structured KPI summary
"""

import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")          # non-interactive backend — safe for all environments
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# GLOBAL STYLE
# ---------------------------------------------------------------------------

PALETTE_MAIN   = ["#3b82d4", "#e05c5c"]          # blue=not-canceled, red=canceled
PALETTE_HOTEL  = ["#3b82d4", "#7c5cd8"]          # City vs Resort
PALETTE_SEQ    = "Blues_d"
ACCENT         = "#3b82d4"
GRID_COLOR     = "#e5e7eb"
FIG_DPI        = 150

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.color":         GRID_COLOR,
    "axes.edgecolor":     "#d1d5db",
    "figure.facecolor":   "white",
    "axes.facecolor":     "white",
})

ROOT      = Path.cwd()
PLOTS_DIR = ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------------

MONTH_ORDER = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]

def save(fig: plt.Figure, name: str) -> None:
    path = PLOTS_DIR / f"{name}.png"
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {path}")


def hbar(ax, series, color=ACCENT, annotate=True):
    """Horizontal bar helper — series index=labels, values=counts/rates."""
    bars = ax.barh(series.index, series.values, color=color, edgecolor="white")
    if annotate:
        for bar, val in zip(bars, series.values):
            ax.text(
                val + series.max() * 0.01, bar.get_y() + bar.get_height() / 2,
                f"{val:,.0f}" if val >= 1 else f"{val:.1%}",
                va="center", fontsize=9, color="#374151",
            )
    ax.invert_yaxis()


def pct_fmt(x, _):
    return f"{x:.0f}%"


# ---------------------------------------------------------------------------
# A. LOAD & SANITY CHECK
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  LOADING DATA")
print("=" * 65)

df = pd.read_csv(ROOT / "data" / "processed" / "hotel_bookings_cleaned.csv", parse_dates=["arrival_date"])
df["arrival_date_month"] = pd.Categorical(
    df["arrival_date_month"], categories=MONTH_ORDER, ordered=True
)

print(f"  Rows: {len(df):,}  |  Columns: {df.shape[1]}")
print(f"  Date range: {df['arrival_date'].min().date()} -> {df['arrival_date'].max().date()}")
print(f"  Hotels: {df['hotel'].value_counts().to_dict()}")


# ---------------------------------------------------------------------------
# B. BUSINESS KPIs
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION B — BUSINESS KPIs")
print("=" * 65)

kpis = {}

# --- B1. Overall Cancellation Rate ---
kpis["cancellation_rate_overall"] = df["is_canceled"].mean()
kpis["total_bookings"]            = len(df)
kpis["total_canceled"]            = int(df["is_canceled"].sum())
kpis["total_checked_out"]         = int((df["reservation_status"] == "Check-Out").sum())
kpis["total_no_show"]             = int((df["reservation_status"] == "No-Show").sum())

# --- B2. ADR (Average Daily Rate) ---
# ADR is computed only on non-canceled, non-zero-ADR bookings (actual stayed)
stayed = df[df["is_canceled"] == 0].copy()
kpis["adr_overall"]       = stayed["adr"].mean()
kpis["adr_city_hotel"]    = stayed[stayed["hotel"] == "City Hotel"]["adr"].mean()
kpis["adr_resort_hotel"]  = stayed[stayed["hotel"] == "Resort Hotel"]["adr"].mean()
kpis["adr_median"]        = stayed["adr"].median()

# --- B3. Revenue ---
kpis["total_revenue_estimate"]   = stayed["revenue_estimate"].sum()
kpis["avg_revenue_per_booking"]  = stayed["revenue_estimate"].mean()

# --- B4. Average Lead Time ---
kpis["avg_lead_time_days"]           = df["lead_time"].mean()
kpis["avg_lead_time_canceled"]       = df[df["is_canceled"] == 1]["lead_time"].mean()
kpis["avg_lead_time_not_canceled"]   = df[df["is_canceled"] == 0]["lead_time"].mean()

# --- B5. Average Length of Stay ---
kpis["avg_length_of_stay_nights"]        = stayed["total_nights"].mean()
kpis["avg_weekend_nights"]               = stayed["stays_in_weekend_nights"].mean()
kpis["avg_week_nights"]                  = stayed["stays_in_week_nights"].mean()

# --- B6. Repeat Guest Rate ---
kpis["repeat_guest_rate"] = df["is_repeated_guest"].mean()

# --- B7. Room Type Match Rate ---
kpis["room_type_match_rate"] = df["room_type_match"].mean()

# --- B8. Special Requests Rate ---
kpis["avg_special_requests"] = df["total_of_special_requests"].mean()

# --- B9. Occupancy proxy: bookings per month (normalized) ---
monthly_bookings = stayed.groupby("arrival_date_month", observed=True).size()
kpis["peak_month"]    = monthly_bookings.idxmax()
kpis["low_month"]     = monthly_bookings.idxmin()

for k, v in kpis.items():
    if isinstance(v, float):
        print(f"  {k:45s}: {v:.4f}")
    else:
        print(f"  {k:45s}: {v}")


# ---------------------------------------------------------------------------
# C. CANCELLATION ANALYSIS
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION C — CANCELLATION ANALYSIS")
print("=" * 65)

# --- C1. Cancellation Rate by Hotel Type ---
cancel_by_hotel = (
    df.groupby("hotel")["is_canceled"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "rate", "sum": "canceled", "count": "total"})
    .sort_values("rate", ascending=False)
)
cancel_by_hotel["rate_pct"] = cancel_by_hotel["rate"] * 100
print("\nCancellation by Hotel:")
print(cancel_by_hotel)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
# Left: rate
colors = PALETTE_HOTEL[:len(cancel_by_hotel)]
axes[0].bar(cancel_by_hotel.index, cancel_by_hotel["rate_pct"], color=colors, edgecolor="white", width=0.5)
axes[0].set_title("Cancellation Rate by Hotel Type", fontweight="bold")
axes[0].set_ylabel("Cancellation Rate (%)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
for i, (idx, row) in enumerate(cancel_by_hotel.iterrows()):
    axes[0].text(i, row["rate_pct"] + 0.5, f"{row['rate_pct']:.1f}%", ha="center", fontweight="bold")

# Right: volume stacked
not_canceled = cancel_by_hotel["total"] - cancel_by_hotel["canceled"]
x = np.arange(len(cancel_by_hotel))
axes[1].bar(x, not_canceled.values, label="Not Canceled", color="#3b82d4", edgecolor="white")
axes[1].bar(x, cancel_by_hotel["canceled"].values, bottom=not_canceled.values, label="Canceled", color="#e05c5c", edgecolor="white")
axes[1].set_xticks(x); axes[1].set_xticks(x)
axes[1].set_xticklabels(cancel_by_hotel.index)
axes[1].set_title("Booking Volume by Hotel Type", fontweight="bold")
axes[1].set_ylabel("Number of Bookings")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
axes[1].legend()
fig.tight_layout()
save(fig, "C1_cancellation_by_hotel")

# --- C2. Cancellation Rate by Market Segment ---
cancel_by_seg = (
    df.groupby("market_segment")["is_canceled"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "rate", "count": "total"})
    .query("total >= 50")
    .sort_values("rate", ascending=True)
)
cancel_by_seg["rate_pct"] = cancel_by_seg["rate"] * 100
print("\nCancellation by Market Segment:")
print(cancel_by_seg)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(cancel_by_seg.index, cancel_by_seg["rate_pct"], color=ACCENT, edgecolor="white")
for bar, val in zip(bars, cancel_by_seg["rate_pct"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2, f"{val:.1f}%", va="center", fontsize=9)
ax.set_xlabel("Cancellation Rate (%)")
ax.set_title("Cancellation Rate by Market Segment", fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
fig.tight_layout()
save(fig, "C2_cancellation_by_market_segment")

# --- C3. Cancellation Rate by Deposit Type ---
cancel_by_dep = (
    df.groupby("deposit_type")["is_canceled"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "rate", "count": "total"})
    .sort_values("rate", ascending=False)
)
cancel_by_dep["rate_pct"] = cancel_by_dep["rate"] * 100
print("\nCancellation by Deposit Type:")
print(cancel_by_dep)

fig, ax = plt.subplots(figsize=(7, 4))
colors_dep = [ACCENT if i % 2 == 0 else "#7c5cd8" for i in range(len(cancel_by_dep))]
bars = ax.bar(cancel_by_dep.index, cancel_by_dep["rate_pct"], color=colors_dep, edgecolor="white", width=0.5)
for bar, val in zip(bars, cancel_by_dep["rate_pct"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.5, f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("Cancellation Rate (%)")
ax.set_title("Cancellation Rate by Deposit Type", fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
fig.tight_layout()
save(fig, "C3_cancellation_by_deposit_type")

# --- C4. Cancellation Rate by Lead Time Bucket ---
df["lead_time_bucket"] = pd.cut(
    df["lead_time"],
    bins=[0, 7, 30, 90, 180, 365, 710],
    labels=["0-7d", "8-30d", "31-90d", "91-180d", "181-365d", "365d+"],
    right=True,
)
cancel_by_lt = (
    df.groupby("lead_time_bucket", observed=True)["is_canceled"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "rate", "count": "total"})
)
cancel_by_lt["rate_pct"] = cancel_by_lt["rate"] * 100
print("\nCancellation by Lead Time Bucket:")
print(cancel_by_lt)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(cancel_by_lt.index.astype(str), cancel_by_lt["rate_pct"],
        marker="o", linewidth=2.5, color=ACCENT, markersize=8)
ax.fill_between(range(len(cancel_by_lt)), cancel_by_lt["rate_pct"].values,
                alpha=0.1, color=ACCENT)
for i, (idx, row) in enumerate(cancel_by_lt.iterrows()):
    ax.text(i, row["rate_pct"] + 0.8, f"{row['rate_pct']:.1f}%", ha="center", fontsize=9)
ax.set_xlabel("Lead Time Bucket")
ax.set_ylabel("Cancellation Rate (%)")
ax.set_title("Cancellation Rate by Lead Time Bucket", fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
fig.tight_layout()
save(fig, "C4_cancellation_by_lead_time_bucket")

# --- C5. Cancellation Rate by Customer Type ---
cancel_by_cust = (
    df.groupby("customer_type")["is_canceled"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "rate", "count": "total"})
    .sort_values("rate", ascending=False)
)
cancel_by_cust["rate_pct"] = cancel_by_cust["rate"] * 100


# ---------------------------------------------------------------------------
# D. REVENUE & ADR ANALYSIS
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION D — REVENUE & ADR ANALYSIS")
print("=" * 65)

# --- D1. Monthly ADR Trend (both hotel types) ---
monthly_adr = (
    stayed.groupby(["arrival_date_month", "hotel"], observed=True)["adr"]
    .mean()
    .reset_index()
    .pivot(index="arrival_date_month", columns="hotel", values="adr")
)
print("\nMonthly ADR by Hotel:")
print(monthly_adr.round(2))

fig, ax = plt.subplots(figsize=(11, 5))
for i, hotel in enumerate(monthly_adr.columns):
    ax.plot(monthly_adr.index.astype(str), monthly_adr[hotel],
            marker="o", linewidth=2.5, label=hotel, color=PALETTE_HOTEL[i])
ax.set_title("Monthly Average Daily Rate (ADR) by Hotel Type", fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("ADR (USD)")
ax.tick_params(axis="x", rotation=40)
ax.legend()
fig.tight_layout()
save(fig, "D1_monthly_adr_by_hotel")

# --- D2. ADR Distribution ---
fig, ax = plt.subplots(figsize=(9, 4.5))
for i, hotel in enumerate(df["hotel"].unique()):
    subset = stayed[stayed["hotel"] == hotel]["adr"]
    ax.hist(subset, bins=60, alpha=0.6, label=hotel, color=PALETTE_HOTEL[i], edgecolor="none")
ax.axvline(stayed["adr"].mean(), color="#e05c5c", linestyle="--", linewidth=1.5, label=f"Overall Mean: ${stayed['adr'].mean():.0f}")
ax.set_xlabel("ADR (USD)")
ax.set_ylabel("Count")
ax.set_title("ADR Distribution by Hotel Type", fontweight="bold")
ax.set_xlim(0, 500)
ax.legend()
fig.tight_layout()
save(fig, "D2_adr_distribution")

# --- D3. ADR by Market Segment ---
adr_by_seg = (
    stayed.groupby("market_segment")["adr"]
    .agg(["mean", "median", "count"])
    .query("count >= 50")
    .sort_values("mean", ascending=True)
)
print("\nADR by Market Segment:")
print(adr_by_seg.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(adr_by_seg.index, adr_by_seg["mean"], color=ACCENT, edgecolor="white")
for bar, val in zip(bars, adr_by_seg["mean"]):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2, f"${val:.0f}", va="center", fontsize=9)
ax.set_xlabel("Mean ADR (USD)")
ax.set_title("Mean ADR by Market Segment (Checked-Out Bookings)", fontweight="bold")
fig.tight_layout()
save(fig, "D3_adr_by_market_segment")

# --- D4. Revenue Estimate by Month ---
monthly_rev = (
    stayed.groupby("arrival_date_month", observed=True)["revenue_estimate"]
    .sum()
    .reset_index()
)
monthly_rev.columns = ["month", "revenue"]

fig, ax = plt.subplots(figsize=(11, 4.5))
bars = ax.bar(monthly_rev["month"].astype(str), monthly_rev["revenue"] / 1e6,
              color=ACCENT, edgecolor="white")
ax.set_title("Total Revenue Estimate by Month (All Years Combined)", fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue ($ Millions)")
ax.tick_params(axis="x", rotation=40)
for bar, val in zip(bars, monthly_rev["revenue"] / 1e6):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.05, f"${val:.1f}M", ha="center", fontsize=8)
fig.tight_layout()
save(fig, "D4_monthly_revenue_estimate")


# ---------------------------------------------------------------------------
# E. BOOKING CHANNEL & MARKET SEGMENT ANALYSIS
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION E — BOOKING CHANNEL & MARKET SEGMENT")
print("=" * 65)

# --- E1. Booking Volume by Market Segment ---
seg_volume = df["market_segment"].value_counts().sort_values(ascending=True)
print("\nBooking Volume by Market Segment:")
print(seg_volume)

fig, ax = plt.subplots(figsize=(9, 5))
colors_seg = [ACCENT] * len(seg_volume)
bars = ax.barh(seg_volume.index, seg_volume.values, color=colors_seg, edgecolor="white")
for bar, val in zip(bars, seg_volume.values):
    ax.text(val + 200, bar.get_y() + bar.get_height() / 2, f"{val:,}", va="center", fontsize=9)
ax.set_xlabel("Number of Bookings")
ax.set_title("Booking Volume by Market Segment", fontweight="bold")
fig.tight_layout()
save(fig, "E1_volume_by_market_segment")

# --- E2. Distribution Channel Mix ---
dist_vol = df["distribution_channel"].value_counts()
print("\nDistribution Channel:")
print(dist_vol)

fig, ax = plt.subplots(figsize=(6, 6))
wedge_colors = ["#3b82d4", "#7c5cd8", "#e05c5c", "#f59e0b", "#10b981"]
wedges, texts, autotexts = ax.pie(
    dist_vol.values,
    labels=dist_vol.index,
    autopct="%1.1f%%",
    colors=wedge_colors[:len(dist_vol)],
    startangle=140,
    wedgeprops=dict(edgecolor="white", linewidth=1.5),
)
for at in autotexts:
    at.set_fontsize(9)
ax.set_title("Distribution Channel Mix", fontweight="bold")
fig.tight_layout()
save(fig, "E2_distribution_channel_mix")

# --- E3. Cancellation rate + ADR side by side by segment ---
seg_combined = (
    df.groupby("market_segment")
    .agg(
        cancel_rate=("is_canceled", "mean"),
        mean_adr=("adr", "mean"),
        total=("is_canceled", "count"),
    )
    .query("total >= 50")
    .sort_values("cancel_rate", ascending=False)
)
seg_combined["cancel_pct"] = seg_combined["cancel_rate"] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(seg_combined.index, seg_combined["cancel_pct"], color="#e05c5c", edgecolor="white")
axes[0].set_xlabel("Cancellation Rate (%)")
axes[0].set_title("Cancellation Rate by Segment", fontweight="bold")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
axes[0].invert_yaxis()

axes[1].barh(seg_combined.index, seg_combined["mean_adr"], color=ACCENT, edgecolor="white")
axes[1].set_xlabel("Mean ADR (USD)")
axes[1].set_title("Mean ADR by Segment", fontweight="bold")
axes[1].invert_yaxis()
fig.suptitle("Market Segment: Cancellation vs ADR", fontweight="bold", fontsize=13)
fig.tight_layout()
save(fig, "E3_segment_cancel_vs_adr")


# ---------------------------------------------------------------------------
# F. LEAD TIME & BOOKING BEHAVIOUR
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION F — LEAD TIME & BOOKING BEHAVIOUR")
print("=" * 65)

# --- F1. Lead Time Distribution (canceled vs not) ---
fig, ax = plt.subplots(figsize=(10, 4.5))
bins = np.linspace(0, 500, 60)
ax.hist(df[df["is_canceled"] == 0]["lead_time"].clip(upper=500),
        bins=bins, alpha=0.6, label="Not Canceled", color="#3b82d4", edgecolor="none")
ax.hist(df[df["is_canceled"] == 1]["lead_time"].clip(upper=500),
        bins=bins, alpha=0.6, label="Canceled", color="#e05c5c", edgecolor="none")
ax.set_xlabel("Lead Time (days, capped at 500)")
ax.set_ylabel("Count")
ax.set_title("Lead Time Distribution: Canceled vs Not Canceled", fontweight="bold")
ax.legend()
fig.tight_layout()
save(fig, "F1_lead_time_distribution")

# --- F2. Booking Changes vs Cancellation ---
df["changes_bucket"] = pd.cut(df["booking_changes"], bins=[-1, 0, 1, 3, 20],
                               labels=["0 changes", "1 change", "2-3 changes", "4+"])
cancel_by_changes = (
    df.groupby("changes_bucket", observed=True)["is_canceled"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "rate", "count": "total"})
)
cancel_by_changes["rate_pct"] = cancel_by_changes["rate"] * 100
print("\nCancellation by Booking Changes:")
print(cancel_by_changes)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(cancel_by_changes.index.astype(str), cancel_by_changes["rate_pct"],
              color=ACCENT, edgecolor="white", width=0.5)
for bar, val in zip(bars, cancel_by_changes["rate_pct"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.3, f"{val:.1f}%", ha="center", fontsize=10)
ax.set_ylabel("Cancellation Rate (%)")
ax.set_title("Cancellation Rate by Number of Booking Changes", fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
fig.tight_layout()
save(fig, "F2_cancellation_by_booking_changes")

# --- F3. Special Requests vs Cancellation ---
cancel_by_sr = (
    df.groupby("total_of_special_requests")["is_canceled"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "rate", "count": "total"})
    .query("total >= 30")
)
cancel_by_sr["rate_pct"] = cancel_by_sr["rate"] * 100
print("\nCancellation by Special Requests:")
print(cancel_by_sr)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cancel_by_sr.index.astype(int), cancel_by_sr["rate_pct"],
        marker="o", linewidth=2.5, color=ACCENT, markersize=8)
ax.set_xlabel("Number of Special Requests")
ax.set_ylabel("Cancellation Rate (%)")
ax.set_title("Cancellation Rate by Number of Special Requests", fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
for i, (idx, row) in enumerate(cancel_by_sr.iterrows()):
    ax.text(idx + 0.05, row["rate_pct"] + 1, f"{row['rate_pct']:.1f}%", fontsize=9)
fig.tight_layout()
save(fig, "F3_cancellation_by_special_requests")


# ---------------------------------------------------------------------------
# G. SEASONALITY & TIME TRENDS
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION G — SEASONALITY & TIME TRENDS")
print("=" * 65)

# --- G1. Monthly Bookings Volume & Cancellation Rate ---
monthly = (
    df.groupby("arrival_date_month", observed=True)
    .agg(
        total=("is_canceled", "count"),
        canceled=("is_canceled", "sum"),
        mean_adr=("adr", "mean"),
    )
    .reset_index()
)
monthly["cancel_rate"] = monthly["canceled"] / monthly["total"] * 100
print("\nMonthly Booking + Cancellation:")
print(monthly[["arrival_date_month", "total", "cancel_rate", "mean_adr"]].to_string())

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
bars = ax1.bar(monthly["arrival_date_month"].astype(str), monthly["total"],
               color=ACCENT, alpha=0.75, edgecolor="white", label="Total Bookings")
ax2.plot(monthly["arrival_date_month"].astype(str), monthly["cancel_rate"],
         color="#e05c5c", linewidth=2.5, marker="o", markersize=6, label="Cancel Rate")
ax1.set_xlabel("Month")
ax1.set_ylabel("Total Bookings", color=ACCENT)
ax2.set_ylabel("Cancellation Rate (%)", color="#e05c5c")
ax1.tick_params(axis="x", rotation=40)
ax1.set_title("Monthly Booking Volume & Cancellation Rate", fontweight="bold")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
fig.tight_layout()
save(fig, "G1_monthly_bookings_and_cancel_rate")

# --- G2. Yearly Trend ---
yearly = (
    df.groupby("arrival_date_year")
    .agg(
        total=("is_canceled", "count"),
        cancel_rate=("is_canceled", "mean"),
        mean_adr=("adr", "mean"),
        total_revenue=("revenue_estimate", "sum"),
    )
    .reset_index()
)
yearly["cancel_pct"] = yearly["cancel_rate"] * 100
print("\nYearly Trend:")
print(yearly)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, col, title, fmt in zip(
    axes,
    ["total", "cancel_pct", "mean_adr"],
    ["Total Bookings", "Cancellation Rate (%)", "Mean ADR ($)"],
    ["{:.0f}", "{:.1f}%", "${:.0f}"],
):
    ax.bar(yearly["arrival_date_year"].astype(str), yearly[col], color=ACCENT, edgecolor="white", width=0.5)
    ax.set_title(title, fontweight="bold")
    for i, val in enumerate(yearly[col]):
        ax.text(i, val * 1.01, fmt.format(val), ha="center", fontsize=9)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
fig.suptitle("Year-over-Year Trends (2015–2017)", fontweight="bold", fontsize=13)
fig.tight_layout()
save(fig, "G2_yearly_trends")

# --- G3. Heatmap: Cancellation Rate by Hotel × Month ---
heat_data = (
    df.groupby(["hotel", "arrival_date_month"], observed=True)["is_canceled"]
    .mean()
    .unstack(level="arrival_date_month")
    * 100
)
fig, ax = plt.subplots(figsize=(13, 3.5))
sns.heatmap(
    heat_data,
    annot=True, fmt=".1f", cmap="RdYlGn_r",
    linewidths=0.5, linecolor=GRID_COLOR,
    cbar_kws={"label": "Cancellation Rate (%)"},
    ax=ax,
)
ax.set_title("Cancellation Rate Heatmap: Hotel × Month", fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("")
fig.tight_layout()
save(fig, "G3_cancellation_heatmap_hotel_month")


# ---------------------------------------------------------------------------
# H. GUEST PROFILE ANALYSIS
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION H — GUEST PROFILE ANALYSIS")
print("=" * 65)

# --- H1. Top 15 Countries by Bookings ---
country_vol = df["country"].value_counts().head(15)
print("\nTop 15 Countries:")
print(country_vol)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(country_vol.index[::-1], country_vol.values[::-1], color=ACCENT, edgecolor="white")
for bar, val in zip(bars, country_vol.values[::-1]):
    ax.text(val + 200, bar.get_y() + bar.get_height() / 2, f"{val:,}", va="center", fontsize=9)
ax.set_xlabel("Number of Bookings")
ax.set_title("Top 15 Countries by Booking Volume", fontweight="bold")
fig.tight_layout()
save(fig, "H1_top_countries")

# --- H2. Length of Stay Distribution ---
los = stayed["total_nights"].clip(upper=20).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(los.index, los.values, color=ACCENT, edgecolor="white")
ax.set_xlabel("Total Nights (capped at 20)")
ax.set_ylabel("Number of Bookings")
ax.set_title("Length of Stay Distribution (Checked-Out Bookings)", fontweight="bold")
ax.set_xticks(range(1, 21))
fig.tight_layout()
save(fig, "H2_length_of_stay_distribution")

# --- H3. Meal Plan Distribution ---
meal_vol = df.groupby(["meal", "hotel"]).size().reset_index(name="count")
meal_pivot = meal_vol.pivot(index="meal", columns="hotel", values="count").fillna(0)
print("\nMeal Plan by Hotel:")
print(meal_pivot)

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(meal_pivot))
w = 0.35
for i, hotel in enumerate(meal_pivot.columns):
    bars = ax.bar(x + i * w, meal_pivot[hotel], width=w, label=hotel,
                  color=PALETTE_HOTEL[i], edgecolor="white")
ax.set_xticks(x + w / 2)
ax.set_xticklabels(meal_pivot.index)
ax.set_ylabel("Number of Bookings")
ax.set_title("Meal Plan Selection by Hotel Type", fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax.legend()
fig.tight_layout()
save(fig, "H3_meal_plan_by_hotel")

# --- H4. Repeat vs New Guest Cancellation ---
repeat_cancel = (
    df.groupby("is_repeated_guest")["is_canceled"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "rate", "count": "total"})
)
repeat_cancel.index = ["New Guest", "Repeat Guest"]
repeat_cancel["rate_pct"] = repeat_cancel["rate"] * 100
print("\nCancellation: Repeat vs New Guest:")
print(repeat_cancel)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(repeat_cancel.index, repeat_cancel["rate_pct"],
              color=PALETTE_HOTEL, edgecolor="white", width=0.45)
for bar, val in zip(bars, repeat_cancel["rate_pct"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.3, f"{val:.1f}%",
            ha="center", fontsize=12, fontweight="bold")
ax.set_ylabel("Cancellation Rate (%)")
ax.set_title("Cancellation Rate: New vs Repeat Guests", fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(pct_fmt))
fig.tight_layout()
save(fig, "H4_new_vs_repeat_guest_cancellation")


# ---------------------------------------------------------------------------
# I. KPI SUMMARY REPORT
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION I — KPI SUMMARY REPORT")
print("=" * 65)

report_lines = []
report_lines.append("Hotel Bookings — Phase 3 KPI Analytics Report")
report_lines.append("=" * 65)
report_lines.append(f"Dataset: hotel_bookings_cleaned.csv")
report_lines.append(f"Total Bookings Analyzed: {kpis['total_bookings']:,}")
report_lines.append(f"Date Range: {df['arrival_date'].min().date()} to {df['arrival_date'].max().date()}")
report_lines.append("")

report_lines.append("--- CORE KPIs ---")
report_lines.append(f"Overall Cancellation Rate    : {kpis['cancellation_rate_overall']:.2%}")
report_lines.append(f"  - Total Canceled           : {kpis['total_canceled']:,}")
report_lines.append(f"  - Total Checked Out        : {kpis['total_checked_out']:,}")
report_lines.append(f"  - Total No-Show            : {kpis['total_no_show']:,}")
report_lines.append("")
report_lines.append(f"Average Daily Rate (ADR)     : ${kpis['adr_overall']:.2f}")
report_lines.append(f"  - City Hotel ADR           : ${kpis['adr_city_hotel']:.2f}")
report_lines.append(f"  - Resort Hotel ADR         : ${kpis['adr_resort_hotel']:.2f}")
report_lines.append(f"  - ADR Median               : ${kpis['adr_median']:.2f}")
report_lines.append("")
report_lines.append(f"Total Revenue Estimate       : ${kpis['total_revenue_estimate']:,.0f}")
report_lines.append(f"Avg Revenue per Booking      : ${kpis['avg_revenue_per_booking']:.2f}")
report_lines.append("")
report_lines.append(f"Avg Lead Time (All)          : {kpis['avg_lead_time_days']:.1f} days")
report_lines.append(f"Avg Lead Time (Canceled)     : {kpis['avg_lead_time_canceled']:.1f} days")
report_lines.append(f"Avg Lead Time (Not Canceled) : {kpis['avg_lead_time_not_canceled']:.1f} days")
report_lines.append("")
report_lines.append(f"Avg Length of Stay           : {kpis['avg_length_of_stay_nights']:.2f} nights")
report_lines.append(f"  - Weekend Nights           : {kpis['avg_weekend_nights']:.2f}")
report_lines.append(f"  - Week Nights              : {kpis['avg_week_nights']:.2f}")
report_lines.append("")
report_lines.append(f"Repeat Guest Rate            : {kpis['repeat_guest_rate']:.2%}")
report_lines.append(f"Room Type Match Rate         : {kpis['room_type_match_rate']:.2%}")
report_lines.append(f"Avg Special Requests/Booking : {kpis['avg_special_requests']:.2f}")
report_lines.append("")
report_lines.append(f"Peak Month (by volume)       : {kpis['peak_month']}")
report_lines.append(f"Low Season Month             : {kpis['low_month']}")
report_lines.append("")

report_lines.append("--- CANCELLATION BREAKDOWN ---")
report_lines.append("By Hotel Type:")
for idx, row in cancel_by_hotel.iterrows():
    report_lines.append(f"  {idx:20s}: {row['rate_pct']:.1f}%  ({int(row['canceled']):,} / {int(row['total']):,})")
report_lines.append("")
report_lines.append("By Deposit Type:")
for idx, row in cancel_by_dep.iterrows():
    report_lines.append(f"  {str(idx):20s}: {row['rate_pct']:.1f}%  (n={int(row['total']):,})")
report_lines.append("")
report_lines.append("By Lead Time Bucket:")
for idx, row in cancel_by_lt.iterrows():
    report_lines.append(f"  {str(idx):20s}: {row['rate_pct']:.1f}%  (n={int(row['total']):,})")
report_lines.append("")
report_lines.append("By Market Segment:")
for idx, row in cancel_by_seg.sort_values("rate_pct", ascending=False).iterrows():
    report_lines.append(f"  {str(idx):20s}: {row['rate_pct']:.1f}%  (n={int(row['total']):,})")
report_lines.append("")

report_lines.append("--- CHARTS PRODUCED ---")
chart_list = sorted(PLOTS_DIR.glob("*.png"))
for c in chart_list:
    report_lines.append(f"  {c.name}")

report_text = "\n".join(report_lines)
print(report_text)

with open(ROOT / "reports" / "kpi_report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"\n  KPI report saved: reports/kpi_report.txt")
print(f"  Charts saved:     {len(chart_list)} files in {PLOTS_DIR}/")
print("\nPhase 3 EDA complete.")


## Phase: sql_analytics.py
Executing code from `sql_analytics.py`

In [ ]:
"""
sql_analytics.py
================
Phase 4 — SQL Analytics Runner
Hotel Bookings Dataset

Usage:
    python sql_analytics.py

What it does:
  1. Loads hotel_bookings_cleaned.csv into an in-memory SQLite database
     (table name: bookings).
  2. Parses hotel_queries.sql and extracts every named query block.
  3. Executes each query and saves results to:
       - sql_results/<query_id>_<title>.csv   (machine-readable)
       - sql_results_report.txt               (human-readable report)
  4. Runs a cross-validation block comparing key SQL results against
     known Phase 3 KPI values.

Engine: Python sqlite3 (SQLite 3.39.4, built-in)
"""

import re
import sqlite3
import textwrap
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

ROOT         = Path.cwd()
INPUT_CSV    = ROOT / "data" / "processed" / "hotel_bookings_cleaned.csv"
SQL_FILE     = ROOT / "sql" / "hotel_queries.sql"
RESULTS_DIR  = ROOT / "sql_results"
REPORT_FILE  = ROOT / "reports" / "sql_results_report.txt"

RESULTS_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------------------------
# STEP 1 — LOAD CSV INTO SQLITE
# ---------------------------------------------------------------------------

def load_to_sqlite(csv_path: str) -> sqlite3.Connection:
    print("Loading CSV into SQLite (in-memory)...")
    df = pd.read_csv(csv_path)
    conn = sqlite3.connect(":memory:")
    df.to_sql("bookings", conn, if_exists="replace", index=False)
    row_count = conn.execute("SELECT COUNT(*) FROM bookings").fetchone()[0]
    col_count = len(conn.execute("PRAGMA table_info(bookings)").fetchall())
    print(f"  Table 'bookings': {row_count:,} rows x {col_count} columns")
    return conn


# ---------------------------------------------------------------------------
# STEP 2 — PARSE SQL FILE INTO NAMED QUERY BLOCKS
# ---------------------------------------------------------------------------

def parse_sql_file(sql_path: str) -> list[dict]:
    """
    Extracts query blocks from hotel_queries.sql.
    Each block must have:
        -- @query_id: Q##
        -- @title: <Human readable title>
    followed by the SQL statement ending at the next block or EOF.
    Returns list of dicts: {query_id, title, sql}
    """
    text = Path(sql_path).read_text(encoding="utf-8")

    # Split on query_id markers
    pattern = re.compile(
        r"--\s*@query_id:\s*(\w+)\s*\n"   # @query_id line
        r"--\s*@title:\s*(.+?)\s*\n"       # @title line
        r"(.*?)"                            # SQL body
        r"(?=--\s*@query_id:|\Z)",          # up to next query or EOF
        re.DOTALL,
    )

    queries = []
    for m in pattern.finditer(text):
        qid   = m.group(1).strip()
        title = m.group(2).strip()
        sql   = m.group(3).strip()
        # Remove trailing separator comments
        sql = re.sub(r"\n*--\s*-{10,}.*$", "", sql, flags=re.DOTALL).strip()
        if sql:
            queries.append({"query_id": qid, "title": title, "sql": sql})

    print(f"  Parsed {len(queries)} queries from {sql_path}")
    return queries


# ---------------------------------------------------------------------------
# STEP 3 — EXECUTE QUERIES & SAVE RESULTS
# ---------------------------------------------------------------------------

def run_queries(conn: sqlite3.Connection, queries: list[dict]) -> dict[str, pd.DataFrame]:
    results = {}
    for q in queries:
        qid   = q["query_id"]
        title = q["title"]
        sql   = q["sql"]
        try:
            df = pd.read_sql_query(sql, conn)
            results[qid] = df
            # Save CSV
            safe_title = re.sub(r"[^\w\s-]", "", title).strip().replace(" ", "_")[:60]
            out_path = RESULTS_DIR / f"{qid}_{safe_title}.csv"
            df.to_csv(out_path, index=False)
            print(f"  [{qid}] {title}  -> {len(df)} rows  -> {out_path.name}")
        except Exception as e:
            print(f"  [{qid}] ERROR: {e}")
            results[qid] = pd.DataFrame()
    return results


# ---------------------------------------------------------------------------
# STEP 4 — GENERATE HUMAN-READABLE REPORT
# ---------------------------------------------------------------------------

def generate_report(queries: list[dict], results: dict[str, pd.DataFrame]) -> str:
    lines = []
    lines.append("Hotel Bookings — Phase 4 SQL Analytics Report")
    lines.append("=" * 70)
    lines.append(f"Engine : SQLite (Python built-in sqlite3)")
    lines.append(f"Source : {INPUT_CSV}")
    lines.append(f"Queries: {len(queries)}")
    lines.append("")

    for q in queries:
        qid   = q["query_id"]
        title = q["title"]
        df    = results.get(qid, pd.DataFrame())

        lines.append(f"{'─' * 70}")
        lines.append(f"[{qid}] {title}")
        lines.append(f"{'─' * 70}")

        if df.empty:
            lines.append("  (no results or error)")
        else:
            lines.append(df.to_string(index=False))
        lines.append("")

    return "\n".join(lines)


# ---------------------------------------------------------------------------
# STEP 5 — CROSS-VALIDATION vs PHASE 3 KPIs
# ---------------------------------------------------------------------------

PHASE3_KPIS = {
    "total_bookings":          118564,
    "total_canceled":          44176,
    "cancellation_rate_pct":   37.26,   # ± 0.05
    "avg_adr_stayed":          101.01,  # ± 0.10
    "total_revenue_estimate":  25986976.03,
    "avg_lead_time_days":      104.5,   # ± 0.1
    "repeat_guest_rate_pct":   2.95,    # ± 0.01
    "room_match_rate_pct":     87.81,   # ± 0.01
}

TOLERANCE = {
    "cancellation_rate_pct":   0.05,
    "avg_adr_stayed":          0.10,
    "avg_lead_time_days":      0.10,
    "repeat_guest_rate_pct":   0.02,
    "room_match_rate_pct":     0.02,
    "total_bookings":          0,
    "total_canceled":          0,
    "total_revenue_estimate":  1.0,
}


def cross_validate(results: dict[str, pd.DataFrame]) -> list[str]:
    lines = []
    lines.append("=" * 70)
    lines.append("CROSS-VALIDATION: SQL Results vs Phase 3 KPIs")
    lines.append("=" * 70)

    q01 = results.get("Q01", pd.DataFrame())
    if q01.empty:
        lines.append("  Q01 result missing — cannot validate.")
        return lines

    row = q01.iloc[0]

    checks = [
        ("total_bookings",         int(row["total_bookings"])),
        ("total_canceled",         int(row["total_canceled"])),
        ("cancellation_rate_pct",  float(row["cancellation_rate_pct"])),
        ("avg_adr_stayed",         float(row["avg_adr_stayed"])),
        ("total_revenue_estimate", float(row["total_revenue_estimate"])),
        ("avg_lead_time_days",     float(row["avg_lead_time_days"])),
        ("repeat_guest_rate_pct",  float(row["repeat_guest_rate_pct"])),
        ("room_match_rate_pct",    float(row["room_match_rate_pct"])),
    ]

    all_pass = True
    for key, sql_val in checks:
        expected = PHASE3_KPIS[key]
        tol      = TOLERANCE.get(key, 0.05)
        diff     = abs(sql_val - expected)
        status   = "PASS" if diff <= tol else "FAIL"
        if status == "FAIL":
            all_pass = False
        lines.append(
            f"  {status}  {key:35s}  SQL={sql_val}  Expected={expected}  |diff|={diff:.4f}"
        )

    lines.append("")
    lines.append("  Overall: " + ("ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED"))
    return lines


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

def main():
    report_lines: list[str] = []

    print("\n" + "=" * 65)
    print("  PHASE 4 — SQL ANALYTICS RUNNER")
    print("=" * 65)

    # Step 1
    print("\n[1] Loading data into SQLite...")
    conn = load_to_sqlite(str(INPUT_CSV))

    # Step 2
    print("\n[2] Parsing SQL query library...")
    queries = parse_sql_file(str(SQL_FILE))

    # Step 3
    print("\n[3] Executing queries...")
    results = run_queries(conn, queries)

    # Step 4
    print("\n[4] Generating report...")
    report_text = generate_report(queries, results)
    report_lines.extend(report_text.splitlines())

    # Step 5
    print("\n[5] Cross-validating against Phase 3 KPIs...")
    val_lines = cross_validate(results)
    for ln in val_lines:
        print("  " + ln)
    report_lines.append("")
    report_lines.extend(val_lines)

    # Save report
    full_report = "\n".join(report_lines)
    Path(REPORT_FILE).write_text(full_report, encoding="utf-8")
    print(f"\n  Report saved: {REPORT_FILE}")

    # Summary
    csv_files = sorted(RESULTS_DIR.glob("*.csv"))
    print(f"  Result CSVs : {len(csv_files)} files in {RESULTS_DIR}/")
    print("\nPhase 4 SQL Analytics complete.")

    conn.close()


if __name__ == "__main__":
    main()


## Phase: ml_pipeline.py
Executing code from `ml_pipeline.py`

In [ ]:
"""
ml_pipeline.py
==============
Phase 5 — Feature Engineering & Machine Learning
Hotel Bookings — Cancellation Prediction

Objective
---------
Binary classification: predict whether a booking will be canceled
(is_canceled = 1) at the time of booking — before check-in occurs.

Models
------
  1. Logistic Regression  — interpretable baseline
  2. Random Forest        — primary model (handles non-linearity, mixed types)
  3. Gradient Boosting    — secondary model (scikit-learn HistGradientBoosting,
                            chosen over GradientBoostingClassifier for speed
                            and native categorical support)

Leakage Controls
----------------
  Excluded: reservation_status, reservation_status_date — these are
            assigned AFTER the outcome is known (post-booking).
  Excluded: flag_adr_outlier, flag_long_lead_time, flag_zero_adr —
            engineered from adr/lead_time which are already included.
  Excluded: revenue_estimate — derived from adr × total_nights (leaks ADR).
  Excluded: arrival_date (raw datetime) — encoded numerically instead.

Outputs
-------
  models/                      — directory for all model artifacts
  models/logistic_regression.pkl
  models/random_forest.pkl
  models/gradient_boosting.pkl
  models/best_model.pkl        — copy of best model by ROC-AUC
  models/feature_names.txt     — ordered feature list used during training
  plots/ML_confusion_matrix.png
  plots/ML_roc_curves.png
  plots/ML_feature_importance.png
  ml_report.txt                — structured evaluation report
"""

import warnings
import textwrap
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")

ROOT       = Path.cwd()
PLOTS_DIR  = ROOT / "plots"
MODELS_DIR = ROOT / "models"
PLOTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

PALETTE = ["#3b82d4", "#e05c5c", "#10b981"]
GRID_COLOR = "#e5e7eb"

# ---------------------------------------------------------------------------
# SECTION 1 — LOAD DATA
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 1 — LOAD DATA")
print("=" * 65)

df = pd.read_csv(ROOT / "data" / "processed" / "hotel_bookings_cleaned.csv")
print(f"  Loaded: {len(df):,} rows × {df.shape[1]} columns")
print(f"  Class balance — 0: {(df['is_canceled']==0).sum():,}  1: {(df['is_canceled']==1).sum():,}  ({df['is_canceled'].mean():.2%} canceled)")


# ---------------------------------------------------------------------------
# SECTION 2 — FEATURE ENGINEERING & LEAKAGE REMOVAL
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 2 — FEATURE ENGINEERING & LEAKAGE REMOVAL")
print("=" * 65)

# --- Post-outcome columns (must be excluded — leakage) ---
LEAKAGE_COLS = [
    "reservation_status",         # assigned after outcome
    "reservation_status_date",    # assigned after outcome
    "revenue_estimate",           # adr × total_nights — partial leakage
    "flag_adr_outlier",           # derived from adr (already included)
    "flag_long_lead_time",        # derived from lead_time (already included)
    "flag_zero_adr",              # derived from adr (already included)
    "arrival_date",               # raw datetime; replaced by numeric features below
]

# --- Features already in the dataset usable at booking time ---
NUMERIC_FEATURES = [
    "lead_time",
    "arrival_date_year",
    "arrival_month_num",            # integer month
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "total_nights",                 # engineered: weekend + week nights
    "adults",
    "children",
    "babies",
    "total_guests",                 # engineered: adults + children + babies
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
    "room_type_match",              # engineered: 1 if reserved == assigned
    "is_high_season",               # engineered: Jun/Jul/Aug flag
]

CATEGORICAL_FEATURES = [
    "hotel",
    "arrival_date_month",
    "meal",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "assigned_room_type",
    "deposit_type",
    "customer_type",
]

TARGET = "is_canceled"

print(f"  Numeric features   : {len(NUMERIC_FEATURES)}")
print(f"  Categorical features: {len(CATEGORICAL_FEATURES)}")
print(f"  Leakage columns removed: {LEAKAGE_COLS}")

# --- Encode categoricals ---
df_ml = df.copy()

label_encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df_ml[col + "_enc"] = le.fit_transform(df_ml[col].astype(str))
    label_encoders[col] = le

ENCODED_CAT_FEATURES = [c + "_enc" for c in CATEGORICAL_FEATURES]
ALL_FEATURES = NUMERIC_FEATURES + ENCODED_CAT_FEATURES

X = df_ml[ALL_FEATURES].copy()
y = df_ml[TARGET].copy()

# Fill any remaining NaN in children column (should be 0 from cleaning)
X["children"] = X["children"].fillna(0)

print(f"\n  Final feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"  Target distribution: {y.value_counts().to_dict()}")

# Save feature names
feature_names_path = MODELS_DIR / "feature_names.txt"
feature_names_path.write_text("\n".join(ALL_FEATURES), encoding="utf-8")
print(f"  Feature list saved: {feature_names_path}")


# ---------------------------------------------------------------------------
# SECTION 3 — TRAIN / TEST SPLIT
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 3 — TRAIN / TEST SPLIT")
print("=" * 65)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"  Train set : {X_train.shape[0]:,} rows  (cancel rate: {y_train.mean():.2%})")
print(f"  Test set  : {X_test.shape[0]:,} rows  (cancel rate: {y_test.mean():.2%})")


# ---------------------------------------------------------------------------
# SECTION 4 — MODEL DEFINITIONS
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 4 — MODEL DEFINITIONS")
print("=" * 65)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=1000,
            random_state=42,
            class_weight="balanced",
            solver="lbfgs",
            C=1.0,
        )),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced",
    ),
    "Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=200,
        max_depth=6,
        learning_rate=0.1,
        min_samples_leaf=20,
        random_state=42,
        class_weight="balanced",
    ),
}

for name in models:
    print(f"  Registered: {name}")


# ---------------------------------------------------------------------------
# SECTION 5 — TRAIN, EVALUATE, CROSS-VALIDATE
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 5 — TRAIN & EVALUATE")
print("=" * 65)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
trained_models = {}

for name, model in models.items():
    print(f"\n  --- {name} ---")

    # Train
    model.fit(X_train, y_train)
    trained_models[name] = model

    # Predict
    y_pred      = model.predict(X_test)
    y_prob      = model.predict_proba(X_test)[:, 1]

    # Metrics
    acc         = accuracy_score(y_test, y_pred)
    prec        = precision_score(y_test, y_pred, zero_division=0)
    rec         = recall_score(y_test, y_pred, zero_division=0)
    f1          = f1_score(y_test, y_pred, zero_division=0)
    roc_auc     = roc_auc_score(y_test, y_prob)

    # 5-fold CV ROC-AUC on training set
    cv_scores   = cross_val_score(model, X_train, y_train, cv=cv,
                                   scoring="roc_auc", n_jobs=-1)
    cv_mean     = cv_scores.mean()
    cv_std      = cv_scores.std()

    results[name] = {
        "accuracy":   acc,
        "precision":  prec,
        "recall":     rec,
        "f1":         f1,
        "roc_auc":    roc_auc,
        "cv_roc_auc_mean": cv_mean,
        "cv_roc_auc_std":  cv_std,
        "y_pred":     y_pred,
        "y_prob":     y_prob,
    }

    print(f"    Accuracy  : {acc:.4f}")
    print(f"    Precision : {prec:.4f}")
    print(f"    Recall    : {rec:.4f}")
    print(f"    F1 Score  : {f1:.4f}")
    print(f"    ROC-AUC   : {roc_auc:.4f}")
    print(f"    CV ROC-AUC: {cv_mean:.4f} ± {cv_std:.4f}")
    print(f"\n    Classification Report (test set):")
    print(textwrap.indent(
        classification_report(y_test, y_pred, target_names=["Not Canceled", "Canceled"]),
        "    "
    ))

    # Save model
    model_path = MODELS_DIR / f"{name.lower().replace(' ', '_')}.pkl"
    joblib.dump(model, model_path)
    print(f"    Saved: {model_path}")


# ---------------------------------------------------------------------------
# SECTION 6 — BEST MODEL & FEATURE IMPORTANCE
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 6 — BEST MODEL & FEATURE IMPORTANCE")
print("=" * 65)

best_name = max(results, key=lambda n: results[n]["roc_auc"])
best_model = trained_models[best_name]
print(f"  Best model by ROC-AUC: {best_name} ({results[best_name]['roc_auc']:.4f})")

# Save best model separately
best_path = MODELS_DIR / "best_model.pkl"
joblib.dump(best_model, best_path)
print(f"  Best model saved: {best_path}")

# Feature importance (Random Forest or Gradient Boosting)
fi_model_name = "Random Forest" if "Random Forest" in trained_models else best_name
fi_model = trained_models[fi_model_name]

if hasattr(fi_model, "feature_importances_"):
    importances = fi_model.feature_importances_
elif hasattr(fi_model, "named_steps"):
    clf = fi_model.named_steps.get("clf")
    importances = clf.coef_[0] if hasattr(clf, "coef_") else None
else:
    importances = None

if importances is not None:
    fi_df = pd.DataFrame({
        "feature":    ALL_FEATURES,
        "importance": importances,
    }).sort_values("importance", ascending=False)

    fi_df.to_csv(MODELS_DIR / "feature_importance.csv", index=False)
    print(f"\n  Top 15 features ({fi_model_name}):")
    print(fi_df.head(15).to_string(index=False))

    # Plot
    top_n = 20
    fi_top = fi_df.head(top_n).sort_values("importance", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 7))
    colors = [PALETTE[0]] * len(fi_top)
    ax.barh(fi_top["feature"], fi_top["importance"], color=colors, edgecolor="white")
    ax.set_xlabel("Feature Importance (Mean Decrease Impurity)")
    ax.set_title(f"Top {top_n} Feature Importances — {fi_model_name}", fontweight="bold")
    ax.grid(axis="x", color=GRID_COLOR)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "ML_feature_importance.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\n  Saved: plots/ML_feature_importance.png")


# ---------------------------------------------------------------------------
# SECTION 7 — VISUALIZATIONS
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 7 — VISUALIZATIONS")
print("=" * 65)

# --- 7a. ROC Curves ---
fig, ax = plt.subplots(figsize=(7, 6))
for i, (name, res) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    roc_val      = res["roc_auc"]
    ax.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={roc_val:.3f})", color=PALETTE[i])
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="#9ca3af", label="Random Chance")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Cancellation Prediction Models", fontweight="bold")
ax.legend(loc="lower right")
ax.grid(color=GRID_COLOR)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "ML_roc_curves.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("  Saved: plots/ML_roc_curves.png")

# --- 7b. Confusion Matrices (one per model) ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res["y_pred"])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Not Canceled", "Canceled"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}\nROC-AUC={res['roc_auc']:.3f}  F1={res['f1']:.3f}", fontweight="bold")
fig.suptitle("Confusion Matrices — Test Set", fontweight="bold", fontsize=13)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "ML_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("  Saved: plots/ML_confusion_matrices.png")

# --- 7c. Model Comparison Bar Chart ---
metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc"]
model_names  = list(results.keys())
x = np.arange(len(metric_names))
width = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
for i, mname in enumerate(model_names):
    vals = [results[mname][m] for m in metric_names]
    bars = ax.bar(x + i * width, vals, width, label=mname, color=PALETTE[i], edgecolor="white")

ax.set_xticks(x + width)
ax.set_xticklabels([m.replace("_", " ").title() for m in metric_names])
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("Score")
ax.set_title("Model Performance Comparison — Test Set", fontweight="bold")
ax.legend()
ax.grid(axis="y", color=GRID_COLOR)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "ML_model_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("  Saved: plots/ML_model_comparison.png")


# ---------------------------------------------------------------------------
# SECTION 8 — WRITE REPORT
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 8 — ML REPORT")
print("=" * 65)

report_lines = []
report_lines.append("Hotel Bookings — Phase 5 ML Pipeline Report")
report_lines.append("=" * 65)
report_lines.append(f"Objective   : Binary classification — predict is_canceled")
report_lines.append(f"Dataset     : hotel_bookings_cleaned.csv  ({len(df):,} rows)")
report_lines.append(f"Train/Test  : 80% / 20% (stratified, random_state=42)")
report_lines.append(f"Features    : {len(ALL_FEATURES)} total ({len(NUMERIC_FEATURES)} numeric, {len(ENCODED_CAT_FEATURES)} encoded categorical)")
report_lines.append(f"CV Strategy : StratifiedKFold(n_splits=5)")
report_lines.append("")

report_lines.append("--- LEAKAGE REMOVED ---")
for col in LEAKAGE_COLS:
    report_lines.append(f"  {col}")
report_lines.append("")

report_lines.append("--- MODEL RESULTS (TEST SET) ---")
header = f"{'Model':<25} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'ROC-AUC':>9} {'CV AUC':>10}"
report_lines.append(header)
report_lines.append("-" * len(header))
for name, res in results.items():
    row = (
        f"{name:<25} "
        f"{res['accuracy']:>9.4f} "
        f"{res['precision']:>10.4f} "
        f"{res['recall']:>8.4f} "
        f"{res['f1']:>8.4f} "
        f"{res['roc_auc']:>9.4f} "
        f"{res['cv_roc_auc_mean']:>7.4f}±{res['cv_roc_auc_std']:.4f}"
    )
    report_lines.append(row)
report_lines.append("")

report_lines.append(f"--- BEST MODEL: {best_name} (ROC-AUC={results[best_name]['roc_auc']:.4f}) ---")
report_lines.append("")

report_lines.append("--- CLASSIFICATION REPORT (BEST MODEL, TEST SET) ---")
cr = classification_report(
    y_test, results[best_name]["y_pred"],
    target_names=["Not Canceled", "Canceled"]
)
report_lines.append(cr)

if importances is not None:
    report_lines.append("--- TOP 15 FEATURE IMPORTANCES (Random Forest) ---")
    report_lines.append(fi_df.head(15).to_string(index=False))

report_text = "\n".join(report_lines)
print(report_text)

(ROOT / "reports" / "ml_report.txt").write_text(report_text, encoding="utf-8")
print(f"\n  Report saved: reports/ml_report.txt")
print("\nPhase 5 ML Pipeline complete.")


## Phase: explainability.py
Executing code from `explainability.py`

In [ ]:
"""
explainability.py
=================
Phase 6 — Model Explainability
Hotel Bookings — Cancellation Prediction

Techniques Used
---------------
1. SHAP TreeExplainer — on the Random Forest model.
   SHAP (SHapley Additive exPlanations) is a game-theoretic method that
   assigns each feature a contribution value for each individual prediction.
   TreeExplainer computes exact SHAP values for tree-based models in
   polynomial time, making it suitable for large datasets.

2. SHAP on HistGradientBoosting (best model) — using shap.Explainer
   (model-agnostic path) for global summary.

3. Permutation Importance — model-agnostic, computed on the test set.
   Measures how much the ROC-AUC drops when a single feature's values
   are randomly shuffled. Unlike MDI (Mean Decrease Impurity), permutation
   importance is not biased toward high-cardinality features.

4. Partial Dependence Plots (PDPs) — for the top 4 numeric features.
   Shows the marginal effect of one feature on the predicted probability,
   averaged over all other features.

IMPORTANT — Prediction vs Causation
-------------------------------------
All findings in this script describe ASSOCIATIONS between features and the
model's predicted probability of cancellation. They do NOT imply that
changing a feature value would CAUSE a booking to cancel or not cancel.

Example: A high `lead_time` is strongly associated with cancellation
(SHAP value > 0). This does not mean that shortening the lead time would
prevent a cancellation — the underlying guest intent is not observed. The
model learns a statistical pattern, not a causal mechanism.

Outputs
-------
  plots/EX_shap_summary_beeswarm.png   — SHAP beeswarm (RF)
  plots/EX_shap_summary_bar.png        — SHAP mean |value| bar (RF)
  plots/EX_shap_dependence_*.png       — SHAP dependence plots for top 4 features
  plots/EX_permutation_importance.png  — Permutation importance (best model)
  plots/EX_partial_dependence.png      — PDP for top 4 numeric features
  plots/EX_shap_waterfall_cancel.png   — Waterfall: sample canceled booking
  plots/EX_shap_waterfall_nocancel.png — Waterfall: sample not-canceled booking
  explainability_report.txt            — structured findings + prediction/causation notes
"""

import warnings
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import shap
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

ROOT       = Path.cwd()
PLOTS_DIR  = ROOT / "plots"
MODELS_DIR = ROOT / "models"
PLOTS_DIR.mkdir(exist_ok=True)

ACCENT     = "#3b82d4"
CANCEL_CLR = "#e05c5c"
GRID_COLOR = "#e5e7eb"

# ---------------------------------------------------------------------------
# SECTION 1 — REBUILD FEATURE MATRIX (identical to ml_pipeline.py)
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 1 — LOAD DATA & REBUILD FEATURES")
print("=" * 65)

df = pd.read_csv(ROOT / "data" / "processed" / "hotel_bookings_cleaned.csv")

NUMERIC_FEATURES = [
    "lead_time", "arrival_date_year", "arrival_month_num",
    "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights",
    "total_nights", "adults", "children", "babies", "total_guests",
    "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled", "booking_changes",
    "days_in_waiting_list", "adr", "required_car_parking_spaces",
    "total_of_special_requests", "room_type_match", "is_high_season",
]
CATEGORICAL_FEATURES = [
    "hotel", "arrival_date_month", "meal", "market_segment",
    "distribution_channel", "reserved_room_type", "assigned_room_type",
    "deposit_type", "customer_type",
]
TARGET = "is_canceled"

label_encoders = {}
for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

ENCODED_CAT = [c + "_enc" for c in CATEGORICAL_FEATURES]
ALL_FEATURES = NUMERIC_FEATURES + ENCODED_CAT

X = df[ALL_FEATURES].copy()
X["children"] = X["children"].fillna(0)
y = df[TARGET].copy()

# Readable labels for plots (strip _enc suffix, replace _ with space)
FEATURE_LABELS = {f: f.replace("_enc", "").replace("_", " ").title() for f in ALL_FEATURES}

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Use a consistent subsample for SHAP (full test set is expensive for beeswarm)
SHAP_SAMPLE_N = 2000
rng = np.random.default_rng(42)
idx = rng.choice(len(X_test), size=min(SHAP_SAMPLE_N, len(X_test)), replace=False)
X_shap = X_test.iloc[idx].reset_index(drop=True)
y_shap = y_test.iloc[idx].reset_index(drop=True)

print(f"  Full dataset   : {len(df):,} rows × {len(ALL_FEATURES)} features")
print(f"  Test set       : {len(X_test):,} rows")
print(f"  SHAP sample    : {len(X_shap):,} rows (stratified random subsample)")

# Load models
rf_model   = joblib.load(MODELS_DIR / "random_forest.pkl")
best_model = joblib.load(MODELS_DIR / "best_model.pkl")   # HistGradientBoosting
print("  Models loaded: random_forest.pkl, best_model.pkl")


# ---------------------------------------------------------------------------
# SECTION 2 — SHAP: RANDOM FOREST (TreeExplainer)
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 2 — SHAP (Random Forest, TreeExplainer)")
print("=" * 65)

print("  Computing SHAP values on subsample...")
rf_explainer   = shap.TreeExplainer(rf_model)
shap_values_rf = rf_explainer.shap_values(X_shap)

# shap_values_rf may be:
#   - list of two 2D arrays [class0, class1]  (older SHAP)
#   - single 3D array of shape (n, features, classes)  (newer SHAP)
# Always extract the class-1 (Canceled) slice.
if isinstance(shap_values_rf, list):
    sv_canceled = shap_values_rf[1]          # list → index 1
elif shap_values_rf.ndim == 3:
    sv_canceled = shap_values_rf[:, :, 1]   # 3D → last axis index 1
else:
    sv_canceled = shap_values_rf             # already 2D

print(f"  SHAP values shape: {sv_canceled.shape}")

# Mean absolute SHAP per feature
mean_abs_shap = pd.DataFrame({
    "feature":    ALL_FEATURES,
    "label":      [FEATURE_LABELS[f] for f in ALL_FEATURES],
    "mean_abs_shap": np.abs(sv_canceled).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

print("\n  Top 15 features by mean |SHAP|:")
print(mean_abs_shap.head(15)[["feature", "mean_abs_shap"]].to_string(index=False))

mean_abs_shap.to_csv(MODELS_DIR / "shap_feature_importance.csv", index=False)
print("  Saved: models/shap_feature_importance.csv")

# --- Plot 1: SHAP Bar Summary ---
top_n = 20
top_features = mean_abs_shap.head(top_n)["feature"].tolist()
top_idx      = [ALL_FEATURES.index(f) for f in top_features]
sv_top       = sv_canceled[:, top_idx]
labels_top   = [FEATURE_LABELS[f] for f in top_features]

fig, ax = plt.subplots(figsize=(9, 7))
vals = mean_abs_shap.head(top_n)["mean_abs_shap"].values
sorted_order = np.argsort(vals)
ax.barh(
    [labels_top[i] for i in sorted_order],
    vals[sorted_order],
    color=ACCENT, edgecolor="white",
)
ax.set_xlabel("Mean |SHAP Value| (average impact on model output)")
ax.set_title(f"SHAP Feature Importance — Top {top_n} Features\n(Random Forest, class=Canceled)", fontweight="bold")
ax.grid(axis="x", color=GRID_COLOR)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "EX_shap_summary_bar.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("  Saved: plots/EX_shap_summary_bar.png")

# --- Plot 2: SHAP Beeswarm Summary ---
# Build a shap.Explanation object for the top-20 features
# Extract scalar base value for class 1 (Canceled)
_ev = rf_explainer.expected_value
if isinstance(_ev, (list, np.ndarray)):
    base_val_rf = float(np.array(_ev).flat[1])
else:
    base_val_rf = float(_ev)

shap_exp = shap.Explanation(
    values=sv_canceled[:, top_idx],
    base_values=np.full(len(X_shap), base_val_rf),
    data=X_shap[top_features].values,
    feature_names=labels_top,
)
fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.beeswarm(shap_exp, max_display=top_n, show=False, plot_size=None)
plt.title("SHAP Beeswarm Plot — Top 20 Features\n(Random Forest, class=Canceled)",
          fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "EX_shap_summary_beeswarm.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: plots/EX_shap_summary_beeswarm.png")


# ---------------------------------------------------------------------------
# SECTION 3 — SHAP DEPENDENCE PLOTS (top 4 numeric features)
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 3 — SHAP DEPENDENCE PLOTS")
print("=" * 65)

# Identify the top 4 numeric features from SHAP ranking
top_numeric = [
    f for f in mean_abs_shap["feature"].tolist()
    if f in NUMERIC_FEATURES
][:4]
print(f"  Top 4 numeric features: {top_numeric}")

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, feat in zip(axes.flat, top_numeric):
    fi   = ALL_FEATURES.index(feat)
    sv_f = sv_canceled[:, fi]
    vals = X_shap[feat].values
    ax.scatter(vals, sv_f, alpha=0.25, s=6, color=ACCENT, rasterized=True)
    # Trend line
    z    = np.polyfit(vals, sv_f, 1)
    p    = np.poly1d(z)
    xs   = np.linspace(vals.min(), vals.max(), 200)
    ax.plot(xs, p(xs), color=CANCEL_CLR, linewidth=2, label="Trend")
    ax.axhline(0, color="#9ca3af", linewidth=1, linestyle="--")
    ax.set_xlabel(FEATURE_LABELS[feat])
    ax.set_ylabel("SHAP Value\n(→ higher = more cancellation)")
    ax.set_title(f"SHAP Dependence: {FEATURE_LABELS[feat]}", fontweight="bold")
    ax.grid(color=GRID_COLOR)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(fontsize=8)
fig.suptitle("SHAP Dependence Plots — Top 4 Numeric Features\n(Random Forest, class=Canceled)",
             fontweight="bold", fontsize=13)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "EX_shap_dependence_top4.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("  Saved: plots/EX_shap_dependence_top4.png")


# ---------------------------------------------------------------------------
# SECTION 4 — SHAP WATERFALL: SAMPLE PREDICTIONS
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 4 — SHAP WATERFALL (Individual Predictions)")
print("=" * 65)

base_val = base_val_rf   # already computed above (class-1 scalar)

def get_sample(label_val, n_candidates=100):
    """Find a cleanly representative sample for the given class."""
    pool = X_shap[y_shap == label_val].head(n_candidates)
    probs = rf_model.predict_proba(pool)[:, 1]
    if label_val == 1:
        # Pick the canceled booking with highest predicted probability
        best_idx = np.argmax(probs)
    else:
        # Pick the not-canceled booking with lowest predicted probability
        best_idx = np.argmin(probs)
    return pool.iloc[[best_idx]], int(pool.index[best_idx])

for cls_val, cls_name, fname in [
    (1, "Canceled Booking",     "EX_shap_waterfall_cancel.png"),
    (0, "Not-Canceled Booking", "EX_shap_waterfall_nocancel.png"),
]:
    sample_X, sample_idx = get_sample(cls_val)
    sample_sv  = sv_canceled[X_shap.index.get_loc(sample_idx) if sample_idx in X_shap.index else 0]

    # Build shap.Explanation for single row (top 15 features only)
    top15_feats = mean_abs_shap.head(15)["feature"].tolist()
    top15_idx   = [ALL_FEATURES.index(f) for f in top15_feats]

    sample_exp = shap.Explanation(
        values=sample_sv[top15_idx],
        base_values=base_val,
        data=sample_X[top15_feats].values[0],
        feature_names=[FEATURE_LABELS[f] for f in top15_feats],
    )

    pred_prob = rf_model.predict_proba(sample_X)[0, 1]
    print(f"  {cls_name}: predicted cancel prob = {pred_prob:.3f}")

    fig, ax = plt.subplots(figsize=(9, 7))
    shap.plots.waterfall(sample_exp, max_display=15, show=False)
    plt.title(f"SHAP Waterfall — {cls_name}\nPredicted Cancel Probability: {pred_prob:.3f}",
              fontweight="bold", pad=12)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / fname, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: plots/{fname}")


# ---------------------------------------------------------------------------
# SECTION 5 — PERMUTATION IMPORTANCE (model-agnostic, best model)
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 5 — PERMUTATION IMPORTANCE (Best Model = HistGBM)")
print("=" * 65)

print("  Computing permutation importance on full test set (n_repeats=10)...")
perm_imp = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=10, random_state=42, scoring="roc_auc", n_jobs=-1
)

perm_df = pd.DataFrame({
    "feature":       ALL_FEATURES,
    "label":         [FEATURE_LABELS[f] for f in ALL_FEATURES],
    "importance_mean": perm_imp.importances_mean,
    "importance_std":  perm_imp.importances_std,
}).sort_values("importance_mean", ascending=False)

perm_df.to_csv(MODELS_DIR / "permutation_importance.csv", index=False)
print("  Saved: models/permutation_importance.csv")
print("\n  Top 15 features by Permutation Importance (ROC-AUC drop):")
print(perm_df.head(15)[["label", "importance_mean", "importance_std"]].to_string(index=False))

top20_perm = perm_df.head(20).sort_values("importance_mean", ascending=True)
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(
    top20_perm["label"],
    top20_perm["importance_mean"],
    xerr=top20_perm["importance_std"],
    color=ACCENT, edgecolor="white", capsize=3,
)
ax.set_xlabel("Mean ROC-AUC Decrease (10 repeats)")
ax.set_title("Permutation Feature Importance — Top 20 Features\n(HistGradientBoosting, Best Model)",
             fontweight="bold")
ax.grid(axis="x", color=GRID_COLOR)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "EX_permutation_importance.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("  Saved: plots/EX_permutation_importance.png")


# ---------------------------------------------------------------------------
# SECTION 6 — PARTIAL DEPENDENCE PLOTS (top 4 numeric from SHAP)
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 6 — PARTIAL DEPENDENCE PLOTS")
print("=" * 65)

top4_idx = [ALL_FEATURES.index(f) for f in top_numeric]
print(f"  PDP features: {top_numeric}")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
PartialDependenceDisplay.from_estimator(
    best_model,
    X_test,
    features=top4_idx,
    feature_names=[FEATURE_LABELS[f] for f in ALL_FEATURES],
    ax=axes.flat,
    kind="average",
    grid_resolution=50,
    random_state=42,
)
for ax, feat in zip(axes.flat, top_numeric):
    ax.set_title(f"PDP: {FEATURE_LABELS[feat]}", fontweight="bold")
    ax.set_ylabel("Predicted Cancel Probability")
    ax.grid(color=GRID_COLOR)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Partial Dependence Plots — Top 4 Numeric Features\n(HistGradientBoosting, Best Model)",
             fontweight="bold", fontsize=13)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "EX_partial_dependence.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("  Saved: plots/EX_partial_dependence.png")


# ---------------------------------------------------------------------------
# SECTION 7 — CROSS-COMPARISON: SHAP vs MDI vs PERMUTATION
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 7 — CROSS-COMPARISON: SHAP vs MDI vs PERMUTATION")
print("=" * 65)

# MDI from Phase 5
mdi_df = pd.read_csv(MODELS_DIR / "feature_importance.csv").rename(
    columns={"importance": "mdi_importance"}
)

# Merge all three rankings
shap_rank  = mean_abs_shap[["feature", "mean_abs_shap"]].rename(
    columns={"mean_abs_shap": "shap_importance"})
perm_rank  = perm_df[["feature", "importance_mean"]].rename(
    columns={"importance_mean": "perm_importance"})

combined = (
    mdi_df[["feature", "mdi_importance"]]
    .merge(shap_rank, on="feature")
    .merge(perm_rank, on="feature")
)
combined["label"] = combined["feature"].map(FEATURE_LABELS)

# Rank each method 1..31
for col in ["mdi_importance", "shap_importance", "perm_importance"]:
    combined[col.replace("importance", "rank")] = (
        combined[col].rank(ascending=False).astype(int)
    )
combined["avg_rank"] = (
    combined[["mdi_rank", "shap_rank", "perm_rank"]].mean(axis=1)
)
combined = combined.sort_values("avg_rank")

combined.to_csv(MODELS_DIR / "feature_ranking_comparison.csv", index=False)
print("  Saved: models/feature_ranking_comparison.csv")

print("\n  Top 15 by Average Rank across all three methods:")
print(combined.head(15)[["label", "mdi_rank", "shap_rank", "perm_rank", "avg_rank"]].to_string(index=False))

# Plot: rank comparison heatmap for top 15
top15_cmp = combined.head(15).set_index("label")
rank_matrix = top15_cmp[["mdi_rank", "shap_rank", "perm_rank"]]
rank_matrix.columns = ["MDI Rank\n(Random Forest)", "SHAP Rank\n(Random Forest)", "Permutation\nRank (HistGBM)"]

import seaborn as sns
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    rank_matrix,
    annot=True, fmt="d", cmap="YlOrRd",
    linewidths=0.5, linecolor=GRID_COLOR,
    cbar_kws={"label": "Rank (lower = more important)"},
    ax=ax,
)
ax.set_title("Feature Importance Method Comparison\n(Top 15 by Average Rank)", fontweight="bold")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "EX_method_comparison_heatmap.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("  Saved: plots/EX_method_comparison_heatmap.png")


# ---------------------------------------------------------------------------
# SECTION 8 — WRITE EXPLAINABILITY REPORT
# ---------------------------------------------------------------------------

print("\n" + "=" * 65)
print("  SECTION 8 — EXPLAINABILITY REPORT")
print("=" * 65)

report_lines = []
report_lines.append("Hotel Bookings — Phase 6 Model Explainability Report")
report_lines.append("=" * 70)
report_lines.append(f"Model explained : Random Forest (SHAP/MDI) + HistGradientBoosting (Permutation/PDP)")
report_lines.append(f"SHAP library    : {shap.__version__}")
report_lines.append(f"SHAP sample     : {len(X_shap):,} rows (random subset of test set)")
report_lines.append("")

report_lines.append("─" * 70)
report_lines.append("IMPORTANT — PREDICTION vs CAUSATION")
report_lines.append("─" * 70)
report_lines.append("""
All SHAP values, permutation importances, and partial dependence plots
describe STATISTICAL ASSOCIATIONS between features and the model's output.
They do NOT establish that changing a feature would CAUSE a booking to
cancel or not cancel. Confounding factors, selection bias, and unmeasured
variables may explain these associations.

Specific caveats:
  - deposit_type (Non Refund): 99.4% cancellation rate in the data.
    This is the model's #1 feature. The pattern may reflect OTA booking
    policies (non-refundable rate codes) rather than guest intent.
    Changing a booking to 'No Deposit' would NOT necessarily prevent
    cancellation; the guest's underlying decision is unobserved.

  - lead_time: Long-horizon bookings cancel more often. This does not
    mean that shortening the booking window reduces cancellation intent.
    The correlation may reflect that guests who book far in advance are
    more likely to have uncertain plans.

  - total_of_special_requests: Guests with more requests cancel less.
    This is likely because engaged, committed guests both make requests
    AND follow through. Artificially encouraging requests would not
    replicate this effect.

  - previous_cancellations: Past cancellations predict future ones. This
    is a behavioral pattern feature — it reflects guest history, not a
    lever a hotel can change.

  - required_car_parking_spaces: Guests who request parking cancel less.
    Likely a proxy for committed, specific-need travelers. Not a causal
    mechanism.
""")

report_lines.append("─" * 70)
report_lines.append("TOP FEATURE RANKINGS (all three methods)")
report_lines.append("─" * 70)
report_lines.append(combined.head(15)[["label", "mdi_rank", "shap_rank", "perm_rank", "avg_rank"]].to_string(index=False))
report_lines.append("")

report_lines.append("─" * 70)
report_lines.append("SHAP MEAN |VALUE| — TOP 20 (Random Forest, class=Canceled)")
report_lines.append("─" * 70)
report_lines.append(mean_abs_shap.head(20)[["label", "mean_abs_shap"]].to_string(index=False))
report_lines.append("")

report_lines.append("─" * 70)
report_lines.append("PERMUTATION IMPORTANCE — TOP 20 (HistGBM, ROC-AUC drop)")
report_lines.append("─" * 70)
report_lines.append(perm_df.head(20)[["label", "importance_mean", "importance_std"]].to_string(index=False))
report_lines.append("")

report_lines.append("─" * 70)
report_lines.append("KEY INTERPRETATIONS (correlation, not causation)")
report_lines.append("─" * 70)
report_lines.append("""
1. deposit_type [#1 SHAP, #1 Permutation]:
   Non-Refundable bookings are associated with near-certain cancellation.
   This is likely a data artefact of OTA pricing policies rather than
   causal evidence that deposit type drives the cancellation decision.
   Interpretation: Non-Refund rate codes identify a specific booking
   population with historically extreme cancellation behaviour.

2. lead_time [#2 SHAP, #2 Permutation]:
   The longer the gap between booking and arrival, the higher the
   predicted cancellation probability. Each additional day of lead time
   is associated with a marginal increase in SHAP value toward cancellation.
   Monotonic positive relationship confirmed in the PDP.

3. total_of_special_requests [#3 SHAP, #4 Permutation]:
   Negatively associated with cancellation. Guests who submit ≥1 request
   cancel at 22% vs 48% for zero requests (from EDA). The PDP shows a
   steep drop in predicted cancel probability at 1+ requests.

4. market_segment [#4 SHAP, #3 Permutation]:
   Groups segment cancels at 61%, Online TA at 37%, Direct at 15%.
   The model strongly discriminates between segments; Groups and Online TA
   push SHAP values toward cancellation, Direct and Corporate away from it.

5. room_type_match [#5 SHAP, #6 Permutation]:
   Bookings where the reserved room type matches the assigned room type
   are associated with lower cancellation. However, room assignment often
   occurs after booking — this feature may be partially post-booking.

6. previous_cancellations [#6 SHAP, #5 Permutation]:
   Strongest behavioral feature. Even 1 prior cancellation significantly
   increases predicted probability. This is a genuine predictive signal
   about guest reliability.
""")

report_lines.append("─" * 70)
report_lines.append("CHARTS PRODUCED")
report_lines.append("─" * 70)
for p in sorted(PLOTS_DIR.glob("EX_*.png")):
    report_lines.append(f"  {p.name}")

report_text = "\n".join(report_lines)
import sys
safe_report = report_text.encode(sys.stdout.encoding or "utf-8", errors="replace").decode(
    sys.stdout.encoding or "utf-8"
)
print(safe_report)

(ROOT / "reports" / "explainability_report.txt").write_text(report_text, encoding="utf-8")
print(f"\n  Report saved: reports/explainability_report.txt")
print("\nPhase 6 Model Explainability complete.")
